# Capítulo 7 — Serving em Produção: llama.cpp, SGLang e TensorRT-LLM

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aejepsen/ebook-llm-on-premise/blob/main/notebooks/cap07_serving.ipynb)

**Fontes originais:**
- [orca3/llm-model-serving — ch08/llamaCpp.ipynb](https://github.com/orca3/llm-model-serving/tree/main/ch08)
- [orca3/llm-model-serving — ch08/SGLang.ipynb](https://github.com/orca3/llm-model-serving/tree/main/ch08)
- [orca3/llm-model-serving — ch08/TensorRT_LLM.ipynb](https://github.com/orca3/llm-model-serving/tree/main/ch08)

Adaptado e comentado por **Allan Eric Jepsen**.

Créditos aos autores originais do repositório [orca3/llm-model-serving](https://github.com/orca3/llm-model-serving).

Créditos adicionais:
- [llama-cpp-python](https://github.com/abetlen/llama-cpp-python) — API Python para llama.cpp
- [SGLang Get Started Tutorial](https://docs.sglang.ai/basic_usage/offline_engine_api.html) — Código de exemplo SGLang

---
## 7.1 — llama.cpp: Inferência CPU-First com GGUF

O **llama.cpp** é um runtime de inferência em C/C++ puro, otimizado para rodar LLMs em CPU (com suporte opcional a GPU). Ele usa o formato **GGUF** (GPT-Generated Unified Format), que suporta quantização agressiva (Q4_0, Q8_0, etc.).

Vantagens:
- Roda em qualquer hardware (sem GPU obrigatória)
- Quantização extrema (modelos de 8B cabem em 4GB de RAM)
- Ideal para edge computing e dispositivos embarcados
- API compatível com OpenAI

### Instalação

Compilamos o `llama-cpp-python` com suporte a OpenBLAS para aceleração matemática em CPU.

In [ ]:
# Instalação para Linux e Mac com aceleração OpenBLAS
!CMAKE_ARGS="-DGGML_BLAS=ON -DGGML_BLAS_VENDOR=OpenBLAS" pip install llama-cpp-python

### Carregando modelo GGUF do HuggingFace

O `Llama.from_pretrained` baixa automaticamente o modelo GGUF do HuggingFace Hub. Usamos o Qwen3-8B quantizado em Q8_0 (8 bits), que oferece boa qualidade com uso reduzido de memória.

In [ ]:
from llama_cpp import Llama

# Exemplo de carregamento local (descomentado para uso com modelo local)
# llm = Llama(
#       model_path="./models/7B/llama-model.gguf",
#       # n_gpu_layers=-1, # Descomentar para aceleração GPU
#       # seed=1337, # Descomentar para definir seed específica
#       # n_ctx=2048, # Descomentar para aumentar a janela de contexto
# )

In [ ]:
# Carrega modelo Qwen do HuggingFace no formato GGUF
llm = Llama.from_pretrained(
    repo_id="Qwen/Qwen3-8B-GGUF",
    filename="*Q8_0.gguf",
    verbose=False
)

# Executa inferência com a API de alto nível
output = llm(
    "Q: Name the planets in the solar system? A: ",  # Prompt
    max_tokens=32,    # Gera até 32 tokens
    stop=["Q:", "\n"],  # Para antes de gerar uma nova pergunta
    echo=True         # Inclui o prompt na saída
)
print(output)

### API de Chat Completion

O llama-cpp-python também suporta a API de chat completion, compatível com o formato OpenAI. Isso facilita a integração com aplicações existentes.

In [ ]:
# Exemplo de chat completion
response = llm.create_chat_completion(
    messages=[
        {"role": "system", "content": "You are an assistant who perfectly describes images."},
        {
            "role": "user",
            "content": "Describe this image in detail please."
        }
    ]
)

print(response)

---
## 7.2 — SGLang: Motor de Alto Desempenho com RadixAttention

O **SGLang** (Structured Generation Language) é um framework de serving que implementa **RadixAttention**, uma técnica que reutiliza automaticamente o KV cache entre requisições com prefixos compartilhados.

Vantagens:
- RadixAttention para reuso automático de cache
- Suporte nativo a modelos quantizados (AWQ, GPTQ)
- API offline e servidor HTTP
- Excelente throughput em cenários de alta concorrência

### Instalação

In [ ]:
!pip install --upgrade pip
!pip install uv
!uv pip install "sglang[all]>=0.5.3rc0"

### Motor Offline com SGLang

Iniciamos o motor SGLang no modo offline (sem servidor HTTP), carregando o modelo Qwen3-8B-AWQ (quantizado em 4 bits com AWQ).

In [ ]:
# Inicializa o motor offline
import asyncio
import sglang as sgl
import sglang.test.doc_patch
from sglang.utils import async_stream_and_merge, stream_and_merge

llm = sgl.Engine(model_path="Qwen/Qwen3-8B-AWQ")

### Geração em Lote

Enviamos múltiplos prompts de uma vez e obtemos todas as respostas.

In [ ]:
prompts = [
    "Hello, my name is",
    "The president of the United States is",
    "The capital of France is",
    "The future of AI is",
]

sampling_params = {"temperature": 0.8, "top_p": 0.95}

outputs = llm.generate(prompts, sampling_params)
for prompt, output in zip(prompts, outputs):
    print("===============================")
    print(f"Prompt: {prompt}\nTexto gerado: {output['text']}")

### Streaming com SGLang

O SGLang suporta geração com streaming síncrono, retornando tokens conforme são gerados.

In [ ]:
prompts = [
    "Write a short, neutral self-introduction for a fictional character. Hello, my name is",
    "Provide a concise factual statement about France's capital city. The capital of France is",
    "Explain possible future trends in artificial intelligence. The future of AI is",
]

sampling_params = {
    "temperature": 0.2,
    "top_p": 0.9,
}

print("\n=== Testando geração síncrona com streaming ===")

for prompt in prompts:
    print(f"Prompt: {prompt}")
    merged_output = stream_and_merge(llm, prompt, sampling_params)
    print("Texto gerado:", merged_output)
    print()

---
## 7.3 — TensorRT-LLM: Máxima Performance com NVIDIA

O **TensorRT-LLM** é o framework de otimização da NVIDIA que compila modelos LLM em grafos otimizados para GPUs NVIDIA. Ele aplica automaticamente:
- Fusão de operações (kernel fusion)
- Quantização FP8/INT8
- Paralelismo de tensor
- Inflight batching

Vantagens:
- Máximo throughput em GPUs NVIDIA
- Suporte a quantização FP8 nativa
- Integração com Triton Inference Server

> **Requisitos:** GPU NVIDIA com CUDA 12.8+. Recomendado A100 ou superior.

### Instalação

A instalação do TensorRT-LLM requer PyTorch compatível com CUDA 12.8 e a biblioteca `libopenmpi-dev` para paralelismo.

In [ ]:
!pip3 install torch==2.7.1 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128

In [ ]:
!sudo apt-get -y install libopenmpi-dev

In [ ]:
!pip3 install --upgrade pip setuptools && pip3 install tensorrt_llm

### Inferência com TensorRT-LLM

O TensorRT-LLM aceita modelos diretamente do HuggingFace Hub, caminhos locais ou checkpoints quantizados (como FP8 via Model Optimizer). A compilação do grafo TensorRT acontece automaticamente no primeiro uso.

In [ ]:
from tensorrt_llm import LLM, SamplingParams

In [ ]:
# Aceita nome do HuggingFace, caminho local, ou checkpoints quantizados
# Exemplo: nvidia/Llama-3.1-8B-Instruct-FP8 para modelo FP8
llm = LLM(model="TinyLlama/TinyLlama-1.1B-Chat-v1.0")

# Prompts de exemplo
prompts = [
    "Hello, my name is",
    "The capital of France is",
    "The future of AI is",
]

# Parâmetros de amostragem
sampling_params = SamplingParams(temperature=0.8, top_p=0.95)

# Geração em lote
for output in llm.generate(prompts, sampling_params):
    print(
        f"Prompt: {output.prompt!r}, Texto gerado: {output.outputs[0].text!r}"
    )